In [4]:
import json
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-120b"



In [9]:
ROADMAP_SCHEMA = {
    "type": "object",
    "properties": {
        "idea_title": {"type": "string"},
        "phases": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "phase_name": {"type": "string"},
                    "steps": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "milestone": {"type": "string"},
                    "skills_required": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "skill_name": {"type": "string"},
                                "resources": {
                                    "type": "array",
                                    "items": {
                                        "type": "object",
                                        "properties": {
                                            "title": {"type": "string"},
                                            "url": {"type": "string"},
                                        },
                                        "required": ["title", "url"],
                                        "additionalProperties": False,
                                    },
                                },
                            },
                            "required": ["skill_name", "resources"],
                            "additionalProperties": False,
                        },
                    },
                },
                "required": [
                    "phase_name",
                    "steps",
                    "milestone",
                    "skills_required",
                ],
                "additionalProperties": False,
            },
        },
    },
    "required": ["idea_title", "phases"],
    "additionalProperties": False,
}

SYSTEM_PROMPT = """You are an expert project mentor. Given a project or startup idea,
break it into 3-5 sequential phases. For each phase, give:
- concrete implementation steps (not vague advice)
- one clear milestone that marks the phase as complete
- the specific skills/frameworks needed for that phase, each with 1-2 genuinely
  useful resource links (docs, well-known tutorials, or courses)

Be specific to the idea given. Do not pad with generic advice."""


In [7]:
def generate_roadmap(idea_text: str) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": idea_text},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "roadmap",
                "schema": ROADMAP_SCHEMA,
                "strict": True,
            },
        },
    )
    return json.loads(response.choices[0].message.content)


In [10]:
idea = "A web app that helps users track their daily habits and provides insights on their progress."
roadmap = generate_roadmap(idea)
print(roadmap)

{'idea_title': 'Daily Habit Tracker Web App', 'phases': [{'phase_name': 'Requirement Gathering & Design', 'steps': ['Conduct user interviews to identify core habit tracking features.', 'Create user personas and user journey maps.', 'Draft wireframes for habit entry, calendar view, and dashboard.', 'Design high-fidelity mockups and style guide in Figma.', 'Set up Git repository with branch strategy and CI linting.'], 'milestone': 'Approved design spec and initialized repository with CI pipeline.', 'skills_required': [{'skill_name': 'User Research & UX Design', 'resources': [{'title': 'NNGroup – Conducting User Interviews', 'url': 'https://www.nngroup.com/articles/user-interviews/'}, {'title': 'Figma – Getting Started Guide', 'url': 'https://help.figma.com/hc/en-us/articles/360040514273-Get-started-with-Figma'}]}, {'skill_name': 'Git & CI Setup', 'resources': [{'title': 'Pro Git Book (online)', 'url': 'https://git-scm.com/book/en/v2'}, {'title': 'GitHub Actions – CI Basics', 'url': 'http